# 第III部：実用的な高精度手法
## 1次元Euler方程式の流体スキーム比較

**比較対象：4種類の数値流束＋MUSCL＋SSP-RK2**

統合版 `euler1d_three_parts.ipynb` の対応する部を独立したNotebookに分けました。
**他のNotebookを先に実行する必要はありません。** 本冊を新しいカーネルで上から実行してください。
数値流束・初期条件・厳密解・境界条件・時間発展・失敗判定は、既存の共通コードを使います。
今回の変更は教材の分割です。新しい数値解法は追加していません。

[01 古典的手法](01_classical_methods.ipynb) / [02 現代の実用的手法](02_practical_riemann_solvers.ipynb) / [03 実用的な高精度手法](03_high_accuracy_methods.ipynb)

この3分類は精度の低い順ではありません。古典的LWは滑らかな解に対して2次精度です。
第II部の「実用的」は数値流束の位置づけであり、1次精度構成を研究用の最終設定とする意味ではありません。
第III部は同じ流束に高次精度化を組み合わせた構成です。
表示名は `LLF (Rusanov)`、内部識別子は `LLF` のままです。

## 実行方法と共通設定

この版は **リポジトリ一式**を使用します。`notebooks/` と `src/` の配置を保ってください。
数値計算コードは各冊に複製せず、`src/euler1d_core.py` と `src/euler1d_study.py` を共有します。
共通の方程式・初期条件・誤差基準は、本冊だけでも確認できるよう以下に再掲しています。

最初の設定セルで本冊の計算キャッシュを空にするため、他の冊の変数や計算結果には依存しません。
設定変更後はカーネルを再起動して全セルを実行してください。
格子収束試験は `run_convergence`、追加実験は `run_extras`、Shu–Osherは `run_shu_osher`、動画は `run_movies` で切り替えます。
**動画・追加実験・Shu–Osherは初期設定で無効**です。動画は本冊の手法だけを表示します。
図や動画を除いた軽量配布版なので、実行後に出力が表示されます。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/euler1d_study.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("リポジトリ一式を取得し、notebooks/ と src/ の配置を保って実行してください。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from euler1d_core import PROBLEMS, run_unit_tests
from euler1d_study import PARTS, RUN_CACHE, get_run, run_part, summarize, show_table, plot_profiles, plot_pressure_minimum, run_convergence, shu_reference, export_results, make_part_animation

In [ ]:
PART = "high_accuracy"
CFG = {
    "gamma": 1.4,
    "cfl": 0.4,
    "n_basic": 200,
    "n_convergence": (100, 200, 400, 800),
    "limiter": "minmod",
    "wave_speeds": "einfeldt",
    "roe_entropy": 0.1,
    "n_shu_osher": 400,
    "n_reference": (1600, 3200),
    "reference_limiter": "mc",
    "run_convergence": True,
    "run_shu_osher": False,
    "run_extras": False,
    "run_movies": False,
    "movie_problem": "sod",
    "movie_frames": 61,
    "export_results": True,
}
RUN_CACHE.clear()
RESULTS = {}
reference_fine = None
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print(CFG)
OUTPUT_DIR = ROOT / "three_part_outputs" / PART
print("Part:", PARTS[PART]["title"])
print("Output directory:", OUTPUT_DIR)

### 共通の方程式と保存形

無粘性・断熱の理想気体を扱います。量はすべて無次元です。$E$は**単位体積当たりの全エネルギー**、$e$は**単位質量当たりの内部エネルギー**です。

$$
\frac{\partial\boldsymbol U}{\partial t}+\frac{\partial\boldsymbol F}{\partial x}=0,
\qquad
\boldsymbol U=\begin{pmatrix}\rho\\\rho u\\E\end{pmatrix},
\qquad
\boldsymbol F=\begin{pmatrix}\rho u\\\rho u^2+p\\u(E+p)\end{pmatrix}.
$$
$$
E=\frac{p}{\gamma-1}+\frac12\rho u^2,\qquad
c=\sqrt{\frac{\gamma p}{\rho}},\qquad e=\frac{p}{(\gamma-1)\rho}.
$$

特性速度は$u-c,\ u,\ u+c$です。中央の波が接触・エントロピー波を担います。[R1]

コードでは`U.shape == (N, 3)`とし、`U[:, 0]`が密度、`U[:, 1]`が運動量密度、`U[:, 2]`が全エネルギー密度です。
基本変数`W`は$(\rho,u,p)$の順です。境界流束を差し引く更新式を全方式で共有します。

$$
\overline{\boldsymbol U}^{\,n+1}_i=\overline{\boldsymbol U}^{\,n}_i-
\frac{\Delta t}{\Delta x}(\boldsymbol F^*_{i+1/2}-\boldsymbol F^*_{i-1/2}).
$$

ここで格子上の未知量は**点値ではなくセル平均**です。
一様格子上で区分一定の状態を用いる場合、以下のLF/LWの保存形実装は対応する古典的な差分更新になります。

### 共通のテスト問題と初期条件

すべて$\gamma=1.4$の設定です。$W=(\rho,u,p)$と表記します。
矩形波のセル内占有率と正弦波のセル平均を解析的に求め、格子点で初期条件を単純にサンプルする誤差を避けます。

| キー | 領域・時刻 | 初期条件の要点 | 境界 |
|:--|:--|:--|:--|
| `entropy` | $[0,1],\ t=1$ | $\rho=1+0.2\sin(2\pi x),\ u=p=1$ | 周期 |
| `contact` | $[0,1],\ t=0.25$ | $0.25\le x<0.45$で$\rho=2$、他は1。$u=p=1$ | 周期 |
| `contact_static` | $[0,1],\ t=0.25$ | 同じ矩形密度、$u=0,\ p=1$ | 周期 |
| `sod` | $[0,1],\ x_0=0.5,\ t=0.2$ | $W_L=(1,0,1),\ W_R=(0.125,0,0.1)$ | 流出 |
| `double_rarefaction` | $[-0.5,0.5],\ x_0=0,\ t=0.15$ | $W_L=(1,-2,0.4),\ W_R=(1,2,0.4)$ | 流出 |
| `shu_osher` | $[-5,5],\ x_0=-4,\ t=1.8$ | 左は衝撃波通過後、右は$\rho=1+0.2\sin(5x),\ u=0,\ p=1$ | 左固定流入・右流出 |
| `transonic` | $[0,1],\ x_0=0.3,\ t=0.2$ | $W_L=(1,0.75,1),\ W_R=(0.125,0,0.1)$ | 流出 |
| `lax` | $[-0.5,0.5],\ x_0=0,\ t=0.14$ | $W_L=(0.445,0.698,3.528),\ W_R=(0.5,0,0.571)$ | 流出 |

SodとShu–Osherの設定は文献・公式テスト例と対応します。[R4, R5]
Shu–Osherの左状態は$\rho_L=27/7,\ u_L=20\sqrt{1.4}/9,\ p_L=31/3$とし、よく用いられる小数値の丸めを避けています。
$\gamma$を変更した場合、この左状態が同じMach数の衝撃波に対応するとは限りません。

流出境界は端のセルをゴーストセルにコピーする簡単な条件で、完全な無反射境界ではありません。
指定した終了時刻では、Sodなどのリーマン問題の主要波面は境界に達しません。

Shu–Osher・Lax・音速をまたぐ希薄波は後半の追加実験に残しています。Laxはここでは**問題の名称**で、LF/LWだけを使うという意味ではありません。

### 厳密解と比較の基準

エントロピー波と接触不連続面は、そのまま速度$u$で平行移動します。
Sod・二重希薄波・追加のリーマン問題は、星領域の圧力$p_*$を

$$
f_L(p_*)+f_R(p_*)+u_R-u_L=0
$$

から二分法で求めます。希薄波と衝撃波で分岐する$f_K$、自己相似変数$\xi=(x-x_0)/t$における解をコードに明示しています。[R1]
ここでの「厳密解」は厳密なリーマン解の構造に基づく数値評価で、非線形方程式の解法と数値積分の許容誤差は残ります。

**誤差は保存変数をセル平均した厳密解と比較します。** 波面の位置で積分区間を分割してからGauss積分するので、不連続を横切る単純な台形積分は使いません。
速度と圧力の比較基準は$W(\overline U_{\rm exact})$です。これは一般に$\overline{W(U_{\rm exact})}$とは異なります。

図には見やすさのため点値の厳密解を実線で示しますが、誤差表にはセル平均基準を使います。
Shu–Osherは厳密解を用いず、後で別に高解像度の**数値参照解**を計算します。

この実装は**物理的な真空を含むリーマン解には対応していません**。
真空条件$u_R-u_L\ge2(c_L+c_R)/(\gamma-1)$を検出すると明示的に停止します。
ここで使う二重希薄波問題は真空条件を満たさず、中央の密度・圧力は正です。

### 時間刻み・破綻・保存則

時間刻みは各計算で

$$
\Delta t=C_{\rm CFL}\frac{\Delta x}{\max_i(|u_i|+c_i)}
$$

から求めます。**共通なのはCFL数と選択規則であり、時々刻々の$\Delta t$が全方式で同じとは限りません。**
終端で極端に短い最後のステップだけを取ると、LFでは$\Delta x/\Delta t$が急増します。
これを避けるため、残り時間がCFL上限の2ステップ以内になったら残りを1または2等分します。この規則は全方式に共通です。

非周期境界では領域内の総量が変化しても保存則違反とは限りません。$Q=\sum_iU_i\Delta x$として

$$
R_Q=Q(t)-Q(0)-\int_0^t(F_{\rm left}^*-F_{\rm right}^*)\,dt'
$$

を測定します。SSP-RK2の境界流束も段階の重み$1/2,1/2$で積算します。
表の`balance_max`は質量・運動量・全エネルギーの$|R_Q|$の最大値で、無次元の絶対残差です。
保存残差が小さいことだけで解の精度がよいとは判断できません。

`failed`の場合、`t_reached`は**最後に受理した時刻**です。
`failure_trial_end`は失敗した試行ステップの終端時刻で、中間段階での失敗なら実際の検出段階は`failure_reason`に表示されます。
失敗した計算の最終時刻誤差は計算しません。

### 評価指標

密度の誤差は、領域長$L$で規格化した平均$L_1$誤差

$$
\epsilon_1(\rho)=\frac1L\sum_i\Delta x\,|\overline\rho_i-\overline\rho_{i,{\rm exact}}|
$$

です。滑らかな周期波については、基本Fourierモードの振幅比`amp_ratio`と位相誤差`phase_rad`も測定します。
接触面の`rho_overshoot`は$\max(\rho)-2$の正の部分、`rho_undershoot`は$1-\min(\rho)$の正の部分です。
`mixed_length`は$1.1<\rho<1.9$のセル幅の合計で、二つの遷移領域と振動の影響を含む補助指標です。単一の接触面の厳密な10–90%幅ではありません。

図は岡部・伊藤パレット、英語ラベル、線種とマーカーの併用に統一します。
RoeとHLLCがほぼ重なること自体も、このテストでは重要な結果です。

### 計算本体の自己検証
元の9グループの検証を実行します。これは基本的な回帰テストであり、任意の初期条件での正確さ・正値性を保証するものではありません。

In [ ]:
checks = run_unit_tests(CFG["gamma"])
for i, name in enumerate(checks, 1):
    print(f"PASS {i}: {name}")
print("Validation groups:", len(checks))

<a id="part-3"></a>
# 第III部：実用的な高精度手法 — 同じ流束を高次精度化する

**問い：第II部と同じ数値流束を使い、MUSCL＋SSP-RK2を組み合わせると何が変わるか。**

4流束をすべて、基本変数のMUSCL再構築・同じminmodリミッター・SSP-RK2で比較します。
この部の「高精度」は既存の2次精度構成を指し、3次以上の手法を実装したという意味ではありません。
今回はリミッターや時間積分法を総当たりで切り替えません。それは次の学習課題です。

### ここで固定するMUSCL再構築とSSP-RK2

第III部では基本変数$W=(\rho,u,p)$を再構成します。minmodでは

$$
\Delta W_i=\operatorname{minmod}(W_i-W_{i-1},W_{i+1}-W_i),\qquad
W_{i+1/2,L}=W_i+\frac12\Delta W_i,\quad W_{i+1/2,R}=W_{i+1}-\frac12\Delta W_{i+1}.
$$

`minmod`は二つの差分が同符号なら絶対値の小さい方、異符号なら0です。
`mc`は$\operatorname{minmod}((d_-+d_+)/2,2d_-,2d_+)$を用います。
コード中の$\Delta W_i$は**勾配そのものではなく、セル幅を掛けた増分**です。

空間離散化を$L(U)$とすると、SSP-RK2は

$$
U^{(1)}=U^n+\Delta t L(U^n),\qquad
U^{n+1}=\frac12U^n+\frac12\left[U^{(1)}+\Delta t L(U^{(1)})\right].
$$

各段階で境界条件を更新します。標準CFL数は0.4です。
MUSCLは滑らかな領域で2次精度を目指す一方、極値・不連続近傍ではリミッターが局所精度を下げます。
したがって、観測した収束次数が常に厳密に2になるとは限りません。
また、成分別のリミッターをEuler系に使っただけで、解全体のTVD性や正値性を無条件に保証したことにはなりません。
SSP型時間積分については[R8]を参照してください。

### III.1 滑らかなエントロピー波

第II部と同じ格子数で波形の振幅と位相を比較します。高次精度化後も流束ごとの違いが残るか確認してください。

In [ ]:
RESULTS[("high_accuracy", "entropy")] = run_part("high_accuracy", "entropy", CFG)
runs = RESULTS[("high_accuracy", "entropy")]
plot_profiles(runs, fields=('rho',), logarithmic=False, title_suffix=PARTS["high_accuracy"]["title"])
summarize(runs)

### III.2 静止する接触面

高次精度化によって、もともと保たれていた静止状態が崩れていないかを回帰テストとして確認します。

In [ ]:
RESULTS[("high_accuracy", "contact_static")] = run_part("high_accuracy", "contact_static", CFG)
runs = RESULTS[("high_accuracy", "contact_static")]
plot_profiles(runs, fields=('rho',), logarithmic=False, title_suffix=PARTS["high_accuracy"]["title"])
summarize(runs)

### III.3 移動する接触面

接触面の広がりとovershootを第II部と比較します。不連続面そのものに滑らかな関数の2次精度を要求するわけではありません。

In [ ]:
RESULTS[("high_accuracy", "contact")] = run_part("high_accuracy", "contact", CFG)
runs = RESULTS[("high_accuracy", "contact")]
plot_profiles(runs, fields=('rho',), logarithmic=False, title_suffix=PARTS["high_accuracy"]["title"])
summarize(runs)

### III.4 Sod衝撃波管

再構築と時間積分を共通化したうえで4流束を比較します。第II部から解像度が改善する場所を特定してください。

In [ ]:
RESULTS[("high_accuracy", "sod")] = run_part("high_accuracy", "sod", CFG)
runs = RESULTS[("high_accuracy", "sod")]
plot_profiles(runs, fields=('rho', 'u', 'p'), logarithmic=False, title_suffix=PARTS["high_accuracy"]["title"])
summarize(runs)

### III.5 二重希薄波

高次精度化だけで正値性が保証されたとは限りません。空間再構築・RK中間状態・最終状態の検査結果を確認します。

In [ ]:
RESULTS[("high_accuracy", "double_rarefaction")] = run_part("high_accuracy", "double_rarefaction", CFG)
runs = RESULTS[("high_accuracy", "double_rarefaction")]
plot_profiles(runs, fields=('rho', 'p'), logarithmic=True, title_suffix=PARTS["high_accuracy"]["title"])
summarize(runs)
plot_pressure_minimum(runs)

### III.6 高次精度化後の収束次数

リミッターは極値近傍で再構築を制限します。有限の格子数で観測次数がちょうど2にならない場合も、誤差と次数の両方を記録してください。

In [ ]:
if CFG["run_convergence"]:
    convergence_III = run_convergence("high_accuracy", CFG)

<a id="bridge"></a>
## 部をまたぐ比較：同じHLLCを1次精度とMUSCL＋SSP-RK2で比べる

流束を固定することで、「数値流束を変えた効果」と「高次精度化した効果」を混同しにくくします。
**ここでは空間再構築と時間積分を同時に変更しています。差を空間だけ、または時間だけの効果に帰属させることはできません。**
この切り分けを次の学習で扱います。

ここで必要なHLLCの1次精度解とMUSCL＋SSP-RK2解は、**この冊の中で両方を計算**します。第II部の実行結果やファイルは読み込みません。

In [ ]:
bridge = [get_run("contact", "HLLC", CFG, order=order) for order in (1, 2)]
plot_profiles(bridge, xlim=(0.42, 0.82), title_suffix="Same HLLC flux: first order vs MUSCL + SSP-RK2")
summarize(bridge)

## 任意：本冊の時間変化

`CFG["run_movies"] = True` で、`movie_problem` の問題を本冊の手法だけで比較します。初期値はSodです。
元の `FrameRecorder` を使い、フレーム出力のために数値積分の時間刻みを変更しません。
記録の有無で最終状態・ステップ数・境界流束が一致することを検査します。
**失敗した方式は停止時刻以降に描画しません。** 表示はJavaScriptアニメーションで、ffmpegは不要です。
この簡易動画はShu–Osherには対応しません。従来のStage 1 MP4生成Notebookは別に残しています。

In [ ]:
if CFG["run_movies"]:
    fig, movie = make_part_animation(PART, CFG["movie_problem"], CFG, n_frames=CFG["movie_frames"])
    display(HTML(movie.to_jshtml()))
    plt.close(fig)

## 任意：Shu–Osher問題

`CFG["run_shu_osher"] = True` で有効です。元の初期条件と2段階の高解像度数値参照解を用います。
黒線は**数値参照解であり厳密解ではありません**。粗い参照解と細かい参照解の差も表示します。
参照解を含め、必要な計算は本冊だけで実行します。図の比較対象は本冊の手法に限定します。
高解像度参照解は計算量が多いため、基本演習の後に実行してください。

In [ ]:
if CFG["run_shu_osher"]:
    reference_fine = shu_reference(CFG)
    runs = run_part(PART, "shu_osher", CFG, n=CFG["n_shu_osher"])
    RESULTS[(PART, "shu_osher")] = runs
    plot_profiles(runs, reference=reference_fine, xlim=(-1.0, 2.3), title_suffix=PARTS[PART]["title"])
    summarize(runs, reference=reference_fine)

## 任意：追加実験

`CFG["run_extras"] = True` で有効です。
Lax衝撃波管を本冊の手法で比較します。
Laxは**問題名**で、使うスキームをLF/LWに限定する意味ではありません。

In [ ]:
if CFG["run_extras"]:
    runs = run_part(PART, "lax", CFG)
    RESULTS[(PART, "lax")] = runs
    plot_profiles(runs, title_suffix=PARTS[PART]["title"])
    summarize(runs)

## 結果の保存

本冊の計算結果を `three_part_outputs/` の **本冊専用サブフォルダー**に保存し、他の冊の結果を上書きしません。
同じ冊を同じ設定で実行し直すと、その冊の保存ファイルを更新します。
CSVに内部識別子 `solver`、表示名 `method`、学習分類 `part` を分けて記録します。
NPZの状態配列の番号はCSVの行順に対応します。停止した計算の誤差は最終時刻の誤差として扱いません。
参照解や部をまたぐ補助比較を有効にした場合は、それらの計算も本冊の出力に含みます。

In [ ]:
if CFG["export_results"]:
    saved_rows = export_results(OUTPUT_DIR, CFG, reference=reference_fine)
    failed_rows = [r for r in saved_rows if r["status"] != "ok"]
    print("Distinct runs:", len(saved_rows), "Recorded failures/incomplete runs:", len(failed_rows))
    show_table(failed_rows, ("part", "problem", "method", "order", "t_reached", "failure_trial_end", "failure_p_min"))

<a id="next-study"></a>
# 次の学習：空間再構築と時間積分を切り分ける（この版では未実装）

今回は「どの種類の方法を比べているか」を整理するところまでです。
次は一度に変える要素を一つにします。以下は学習計画で、追加の実装・実験結果ではありません。

| 比較 | 固定するもの | 変えるもの | 主な問題 |
|:--|:--|:--|:--|
| 空間再構築 | 数値流束、時間積分、CFL条件 | 区分一定、MUSCLのリミッター、さらにPPM/WENO等 | エントロピー波、接触面、Shu–Osher |
| 時間積分 | 数値流束、空間再構築、空間格子 | 前進Euler、SSP-RK2、さらにSSP-RK3等 | 滑らかな周期波 |
| 計算コスト | 誤差の目標、物理問題 | 必要な格子数・ステップ数・実行時間 | 滑らかな波と衝撃波問題を別々に評価 |

時間積分の次数を測る場合には空間誤差が支配していないかも確認します。
空間格子を十分細かくし、時間刻みを細かくしても誤差が下がらない領域を見分ける必要があります。
時間積分だけを高次にしても、空間離散化を含めた誤差全体がその次数になるとは限りません。

## この教材の範囲

1次元・一様格子・単一の理想気体が対象です。数値流束が研究用コードで使われる形式でも、この学習用実装全体を研究用に検証済みとみなさないでください。
重力・冷却・MHD・相対論・多次元・真空・包括的な耐破綻処理は追加していません。
元の教材と同様に、負圧をfloorや自動ソルバー切替で隠しません。

## 参考文献・実装の確認先（従来版から継承）

以下は原論文または開発者による公式資料です。Notebookのコードはこれらの数式・考え方を参考に、本教材用に組み立てた実装であり、Clawpack等を呼び出してはいません。

**[R1]** Clawpack, *The Euler equations of gas dynamics*：波の構造と厳密リーマン解。  
https://www.clawpack.org/riemann_book/html/Euler.html

**[R2]** Clawpack, *Approximate solvers for the Euler equations of gas dynamics*：Roe平均、HLLEの波速推定、低密度問題。  
https://www.clawpack.org/riemann_book/html/Euler_approximate.html

**[R3]** Clawpack, *Riemann Solver Package — Euler Equations*：HLLCの接触波速度と中間状態。  
https://www.clawpack.org/pyclaw/rp.html#euler-equations

**[R4]** FLASH User's Guide, *Hydrodynamics Test Problems*, §35.1.1：Sod問題と波面の位置。  
https://flash.rochester.edu/site/flashcode/user_support/flash_ug_devel/node191.html

**[R5]** Clawpack Gallery, *Shu–Osher problem*：初期状態、領域、終了時刻。  
https://www.clawpack.org/gallery/pyclaw/gallery/shocksine.html

**[R6]** Toro, E. F., Spruce, M., & Speares, W. (1994), *Restoration of the contact surface in the HLL-Riemann solver*, Shock Waves, 4, 25–34.  
https://doi.org/10.1007/BF01414629

**[R7]** Harten, A. (1983), *High resolution schemes for hyperbolic conservation laws*, Journal of Computational Physics, 49, 357–393.  
https://doi.org/10.1016/0021-9991(83)90136-5

**[R8]** Shu, C.-W., & Osher, S. (1988), *Efficient implementation of essentially non-oscillatory shock-capturing schemes*, Journal of Computational Physics, 77, 439–471.  
https://doi.org/10.1016/0021-9991(88)90177-5

**[R9]** Shu, C.-W., & Osher, S. (1989), *Efficient implementation of essentially non-oscillatory shock-capturing schemes, II*, Journal of Computational Physics, 83, 32–78.  
https://doi.org/10.1016/0021-9991(89)90222-2

初期条件・波速推定・エントロピー補正・リミッター・CFL数まで含めて、比較の設定を報告することが重要です。
「Roe」や「HLLC」という名前だけでは、実装のすべては決まりません。